# LightGBM Per-Day Features

Instead of flattening to summary stats, feed all 14×41 = 574 individual day values + 27 survey = **601 features** directly to LightGBM.

LightGBM handles NaN natively — no imputation needed.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report
from pathlib import Path

from config import (RANDOM_STATE, N_FOLDS, LABEL_MAP_4_TO_3, LABEL_MAP_4_TO_2,
                    CLASS_NAMES_4, CLASS_NAMES_3, CLASS_NAMES_2, set_seed)

FEATURES_DIR = Path('../outputs/features')
set_seed()
print('Ready')

Ready


## Step 1: Build Per-Day Feature Matrix

In [2]:
X_orig = np.load(FEATURES_DIR / 'X.npy')           # (1586, 14, 22)
X_c    = np.load(FEATURES_DIR / 'X_option_c.npy')   # (1586, 14, 19)
X_survey = np.load(FEATURES_DIR / 'X_survey.npy')   # (1586, 27)
y      = np.load(FEATURES_DIR / 'y.npy')
lengths = np.load(FEATURES_DIR / 'lengths.npy')

X_wear = np.concatenate([X_orig, X_c], axis=2)      # (1586, 14, 41)
N, T, F = X_wear.shape

# Flatten: (N, 14, 41) -> (N, 574)  keeping day structure
X_perday = X_wear.reshape(N, T * F)

# Build feature names: d0_f0, d0_f1, ..., d13_f40
perday_names = [f'd{d}_f{f}' for d in range(T) for f in range(F)]

# Also add summary stats for the most important features (from prev notebook)
# This gives LGB both raw per-day AND aggregate views
def add_summary_stats(X_3d, lengths):
    N, T, F = X_3d.shape
    summaries = []
    names = []
    for f in range(F):
        col = X_3d[:, :, f]
        summaries.append(np.nanmean(col, axis=1))
        summaries.append(np.nanstd(col, axis=1))
        # Slope
        slope = np.zeros(N)
        for i in range(N):
            valid = col[i, :int(lengths[i])]
            valid = valid[~np.isnan(valid)]
            if len(valid) >= 2:
                slope[i] = np.polyfit(np.arange(len(valid)), valid, 1)[0]
        summaries.append(slope)
        names.extend([f'agg_f{f}_mean', f'agg_f{f}_std', f'agg_f{f}_slope'])
    return np.column_stack(summaries), names

X_agg, agg_names = add_summary_stats(X_wear, lengths)

SURVEY_NAMES = [
    'age', 'education_years', 'smoking_ever', 'smoking_now', 'alcohol_ever',
    'cesd_score', 'sleep_restless', 'paid_score', 'sleeping_pills',
    'diet_score', 'diet_fast_food', 'diet_beans', 'diet_regular_food',
    'diet_desserts', 'diet_fats', 'fh_diabetes_parent', 'fh_diabetes_sibling',
    'has_hypertension', 'has_obesity', 'has_high_cholesterol',
    'has_heart_attack', 'has_stroke', 'has_kidney_problems', 'has_circulation',
    'vision_difficulty', 'food_insecurity_1', 'food_insecurity_2'
]

# Combine: per-day + aggregates + survey + seq_length
X_all = np.concatenate([X_perday, X_agg, X_survey, lengths.reshape(-1,1).astype(float)], axis=1)
all_names = perday_names + agg_names + SURVEY_NAMES + ['seq_length']

print(f'Per-day:    {X_perday.shape[1]} features')
print(f'Aggregates: {X_agg.shape[1]} features')
print(f'Survey:     {X_survey.shape[1]} features')
print(f'Total:      {X_all.shape[1]} features')
print(f'NaN ratio:  {np.isnan(X_all).mean()*100:.1f}%')

C:\Users\hp\AppData\Local\Temp\ipykernel_28732\3158848990.py:24: RuntimeWarning: Mean of empty slice
  summaries.append(np.nanmean(col, axis=1))
C:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


Per-day:    574 features
Aggregates: 123 features
Survey:     27 features
Total:      725 features
NaN ratio:  18.1%


## Step 2: Metrics + Training Functions

In [3]:
def compute_all_metrics(y4, proba4, label=''):
    pred = np.argmax(proba4, axis=1)
    acc = accuracy_score(y4, pred)
    auc4 = roc_auc_score(y4, proba4, multi_class='ovr', average='macro')
    y3 = np.array([LABEL_MAP_4_TO_3[i] for i in y4])
    p3 = np.zeros((len(y4), 3))
    p3[:, 0] = proba4[:, 0]; p3[:, 1] = proba4[:, 1]
    p3[:, 2] = proba4[:, 2] + proba4[:, 3]
    auc3 = roc_auc_score(y3, p3, multi_class='ovr', average='macro')
    y2 = np.array([LABEL_MAP_4_TO_2[i] for i in y4])
    auc2 = roc_auc_score(y2, proba4[:, 1] + proba4[:, 2] + proba4[:, 3])
    print(f'\n  {label}')
    print(f'  acc={acc*100:.1f}%  4-AUC={auc4:.4f}  3-AUC={auc3:.4f}  2-AUC={auc2:.4f}')
    print(classification_report(y4, pred, target_names=CLASS_NAMES_4, digits=3))
    return acc, auc4, auc3, auc2

def run_lgb_4class(X, y, names, label, params=None):
    set_seed()
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    oof = np.zeros((len(y), 4))
    importances = np.zeros(X.shape[1])
    if params is None:
        params = {
            'objective': 'multiclass', 'num_class': 4, 'metric': 'multi_logloss',
            'boosting_type': 'gbdt', 'n_estimators': 2000, 'learning_rate': 0.02,
            'max_depth': 6, 'num_leaves': 31, 'min_child_samples': 20,
            'subsample': 0.8, 'colsample_bytree': 0.6, 'reg_alpha': 0.1,
            'reg_lambda': 1.0, 'random_state': RANDOM_STATE, 'verbose': -1,
            'is_unbalance': True,
        }
    print(f'\n[{label}] {X.shape[1]} features')
    for fold, (tr, va) in enumerate(skf.split(X, y)):
        model = lgb.LGBMClassifier(**params)
        model.fit(X[tr], y[tr], eval_set=[(X[va], y[va])],
                  callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
        oof[va] = model.predict_proba(X[va])
        importances += model.feature_importances_
        print(f'  Fold {fold+1}: {model.best_iteration_} rounds')
    importances /= N_FOLDS
    acc, auc4, auc3, auc2 = compute_all_metrics(y, oof, label)
    if names:
        idx = np.argsort(importances)[::-1][:25]
        print('  Top-25 features:')
        for i in idx:
            print(f'    {names[i]:35s}  {importances[i]:.0f}')
    return oof, acc, auc4, auc3, auc2, importances

def run_lgb_3class(X, y4, label):
    y3 = np.array([LABEL_MAP_4_TO_3[i] for i in y4])
    set_seed()
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    oof = np.zeros((len(y3), 3))
    params = {
        'objective': 'multiclass', 'num_class': 3, 'metric': 'multi_logloss',
        'boosting_type': 'gbdt', 'n_estimators': 2000, 'learning_rate': 0.02,
        'max_depth': 6, 'num_leaves': 31, 'min_child_samples': 20,
        'subsample': 0.8, 'colsample_bytree': 0.6, 'reg_alpha': 0.1,
        'reg_lambda': 1.0, 'random_state': RANDOM_STATE, 'verbose': -1,
        'is_unbalance': True,
    }
    print(f'\n[{label}] {X.shape[1]} features')
    for fold, (tr, va) in enumerate(skf.split(X, y3)):
        model = lgb.LGBMClassifier(**params)
        model.fit(X[tr], y3[tr], eval_set=[(X[va], y3[va])],
                  callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
        oof[va] = model.predict_proba(X[va])
        print(f'  Fold {fold+1}: {model.best_iteration_} rounds')
    auc3 = roc_auc_score(y3, oof, multi_class='ovr', average='macro')
    pred = np.argmax(oof, axis=1)
    acc = accuracy_score(y3, pred)
    print(f'\n  {label}: acc={acc*100:.1f}%  3-AUC={auc3:.4f}')
    print(classification_report(y3, pred, target_names=CLASS_NAMES_3, digits=3))
    return oof, auc3

def run_lgb_2class(X, y4, names, label):
    y2 = np.array([LABEL_MAP_4_TO_2[i] for i in y4])
    set_seed()
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    oof = np.zeros(len(y2))
    importances = np.zeros(X.shape[1])
    params = {
        'objective': 'binary', 'metric': 'binary_logloss',
        'boosting_type': 'gbdt', 'n_estimators': 2000, 'learning_rate': 0.02,
        'max_depth': 6, 'num_leaves': 31, 'min_child_samples': 20,
        'subsample': 0.8, 'colsample_bytree': 0.6, 'reg_alpha': 0.1,
        'reg_lambda': 1.0, 'random_state': RANDOM_STATE, 'verbose': -1,
        'is_unbalance': True,
    }
    print(f'\n[{label}] {X.shape[1]} features')
    for fold, (tr, va) in enumerate(skf.split(X, y2)):
        model = lgb.LGBMClassifier(**params)
        model.fit(X[tr], y2[tr], eval_set=[(X[va], y2[va])],
                  callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
        oof[va] = model.predict_proba(X[va])[:, 1]
        importances += model.feature_importances_
        print(f'  Fold {fold+1}: {model.best_iteration_} rounds')
    auc2 = roc_auc_score(y2, oof)
    pred = (oof > 0.5).astype(int)
    acc = accuracy_score(y2, pred)
    print(f'\n  {label}: acc={acc*100:.1f}%  2-AUC={auc2:.4f}')
    print(classification_report(y2, pred, target_names=CLASS_NAMES_2, digits=3))
    importances /= N_FOLDS
    if names:
        idx = np.argsort(importances)[::-1][:25]
        print('  Top-25 features (binary):')
        for i in idx:
            print(f'    {names[i]:35s}  {importances[i]:.0f}')
    return oof, auc2

## Step 3: Train All Three

In [4]:
print('=' * 70)
print('VARIANT A: Per-day + Aggregates + Survey (full 725 features)')
print('=' * 70)

oof4_a, acc4_a, auc4_a, auc3_a, auc2_a, imp_a = run_lgb_4class(
    X_all, y, all_names, 'PerDay+Agg+Survey 4-class')
np.save(FEATURES_DIR / 'oof_lgb_perday_4class.npy', oof4_a)

oof3_a, auc3_ded_a = run_lgb_3class(X_all, y, 'PerDay+Agg+Survey 3-class')
np.save(FEATURES_DIR / 'oof_lgb_perday_3class.npy', oof3_a)

oof2_a, auc2_ded_a = run_lgb_2class(X_all, y, all_names, 'PerDay+Agg+Survey 2-class')
np.save(FEATURES_DIR / 'oof_lgb_perday_2class.npy', oof2_a)

VARIANT A: Per-day + Aggregates + Survey (full 725 features)

[PerDay+Agg+Survey 4-class] 725 features


  Fold 1: 161 rounds


  Fold 2: 113 rounds


  Fold 3: 126 rounds


  Fold 4: 103 rounds


...

  Fold 1: 233 rounds


  Fold 2: 156 rounds


  Fold 3: 138 rounds


  Fold 4: 111 rounds


  Fold 5: 172 rounds

  PerDay+Agg+Survey 2-class: acc=73.3%  2-AUC=0.7685
              precision    recall  f1-score   support

     healthy      0.646     0.537     0.587       560
 not_healthy      0.769     0.839     0.802      1026

    accuracy                          0.733      1586
   macro avg      0.707     0.688     0.695      1586
weighted avg      0.725     0.733     0.726      1586

  Top-25 features (binary):
    paid_score                           95
    has_hypertension                     72
    fh_diabetes_parent                   47
    fh_diabetes_sibling                  39
    d7_f16                               35
    agg_f3_std                           35
    agg_f16_mean                         35
    d0_f22                               29
    has_obesity                          29
    d9_f27                               28
    d2_f9                                26
    agg_f28_slope                        26
    d5_f2                                2

In [5]:
print('\n' + '=' * 70)
print('VARIANT B: Per-day + Survey only (no aggregates) — 601 features')
print('=' * 70)

X_noagg = np.concatenate([X_perday, X_survey, lengths.reshape(-1,1).astype(float)], axis=1)
noagg_names = perday_names + SURVEY_NAMES + ['seq_length']

oof4_b, acc4_b, auc4_b, auc3_b, auc2_b, _ = run_lgb_4class(
    X_noagg, y, noagg_names, 'PerDay+Survey 4-class')

oof3_b, auc3_ded_b = run_lgb_3class(X_noagg, y, 'PerDay+Survey 3-class')

oof2_b, auc2_ded_b = run_lgb_2class(X_noagg, y, noagg_names, 'PerDay+Survey 2-class')


VARIANT B: Per-day + Survey only (no aggregates) — 601 features



[PerDay+Survey 4-class] 602 features


  Fold 1: 156 rounds


  Fold 2: 99 rounds


  Fold 3: 132 rounds


...

  Fold 1: 245 rounds


  Fold 2: 182 rounds


  Fold 3: 143 rounds


  Fold 4: 119 rounds


  Fold 5: 147 rounds

  PerDay+Survey 2-class: acc=72.1%  2-AUC=0.7674
              precision    recall  f1-score   support

     healthy      0.622     0.532     0.574       560
 not_healthy      0.763     0.824     0.792      1026

    accuracy                          0.721      1586
   macro avg      0.693     0.678     0.683      1586
weighted avg      0.713     0.721     0.715      1586

  Top-25 features (binary):
    paid_score                           98
    has_hypertension                     80
    fh_diabetes_parent                   46
    d7_f16                               44
    fh_diabetes_sibling                  39
    d6_f31                               32
    d2_f9                                31
    d9_f27                               30
    d0_f29                               29
    d1_f13                               28
    d0_f22                               28
    has_obesity                          27
    d2_f13                               26
  

## Step 4: Ensemble with Hybrid LSTM

In [6]:
hybrid_oof = np.load(FEATURES_DIR / 'oof_hybrid_T2_a0.5.npy')
prev_lgb_oof = np.load(FEATURES_DIR / 'oof_lgb_4class.npy')  # summary-stats LGB

# Pick best perday variant
if auc4_a >= auc4_b:
    best_perday = oof4_a
    best_label = 'PerDay+Agg+Survey'
else:
    best_perday = oof4_b
    best_label = 'PerDay+Survey'

print(f'Best per-day variant: {best_label}')
print(f'\n=== Ensembles: PerDay-LGB + Hybrid LSTM ===')
for w in [0.3, 0.4, 0.5, 0.6, 0.7]:
    ens = w * best_perday + (1 - w) * hybrid_oof
    compute_all_metrics(y, ens, f'PerDay×{w:.1f} + Hybrid×{1-w:.1f}')

print(f'\n=== 3-way Ensemble: PerDay-LGB + Summary-LGB + Hybrid ===')
for wp, ws, wh in [(0.4, 0.2, 0.4), (0.3, 0.3, 0.4), (0.4, 0.3, 0.3), (0.5, 0.2, 0.3)]:
    ens = wp * best_perday + ws * prev_lgb_oof + wh * hybrid_oof
    compute_all_metrics(y, ens, f'PerDay×{wp} + SumLGB×{ws} + Hybrid×{wh}')

Best per-day variant: PerDay+Agg+Survey

=== Ensembles: PerDay-LGB + Hybrid LSTM ===

  PerDay×0.3 + Hybrid×0.7
  acc=46.5%  4-AUC=0.7312  3-AUC=0.7360  2-AUC=0.7730


              precision    recall  f1-score   support

     healthy      0.579     0.671     0.622       560
 prediabetes      0.379     0.263     0.311       410
    oral_med      0.437     0.401     0.418       446
     insulin      0.306     0.435     0.359       170

    accuracy                          0.465      1586
   macro avg      0.425     0.443     0.428      1586
weighted avg      0.458     0.465     0.456      1586


  PerDay×0.4 + Hybrid×0.6
  acc=48.0%  4-AUC=0.7324  3-AUC=0.7378  2-AUC=0.7771
              precision    recall  f1-score   support

     healthy      0.583     0.693     0.633       560
 prediabetes      0.388     0.261     0.312       410
    oral_med      0.447     0.471     0.459       446
     insulin      0.326     0.335     0.330       170

    accuracy                          0.480      1586
   macro avg      0.436     0.440     0.434      1586
weighted avg      0.467     0.480     0.469      1586


  PerDay×0.5 + Hybrid×0.5
  acc=47.7%  4-AUC=0.7

              precision    recall  f1-score   support

     healthy      0.571     0.680     0.621       560
 prediabetes      0.377     0.261     0.308       410
    oral_med      0.439     0.502     0.469       446
     insulin      0.360     0.265     0.305       170

    accuracy                          0.477      1586
   macro avg      0.437     0.427     0.426      1586
weighted avg      0.461     0.477     0.463      1586


  PerDay×0.6 + Hybrid×0.4
  acc=46.9%  4-AUC=0.7301  3-AUC=0.7367  2-AUC=0.7790


              precision    recall  f1-score   support

     healthy      0.562     0.675     0.613       560
 prediabetes      0.366     0.246     0.294       410
    oral_med      0.427     0.518     0.468       446
     insulin      0.354     0.200     0.256       170

    accuracy                          0.469      1586
   macro avg      0.427     0.410     0.408      1586
weighted avg      0.451     0.469     0.452      1586


  PerDay×0.7 + Hybrid×0.3
  acc=47.0%  4-AUC=0.7269  3-AUC=0.7341  2-AUC=0.7778
              precision    recall  f1-score   support

     healthy      0.560     0.680     0.615       560
 prediabetes      0.355     0.244     0.289       410
    oral_med      0.434     0.547     0.484       446
     insulin      0.323     0.118     0.172       170

    accuracy                          0.470      1586
   macro avg      0.418     0.397     0.390      1586
weighted avg      0.446     0.470     0.446      1586


=== 3-way Ensemble: PerDay-LGB + Summary-LGB + H


  PerDay×0.4 + SumLGB×0.2 + Hybrid×0.4
  acc=48.2%  4-AUC=0.7333  3-AUC=0.7402  2-AUC=0.7839
              precision    recall  f1-score   support

     healthy      0.569     0.684     0.621       560
 prediabetes      0.383     0.259     0.309       410
    oral_med      0.443     0.536     0.485       446
     insulin      0.381     0.218     0.277       170

    accuracy                          0.482      1586
   macro avg      0.444     0.424     0.423      1586
weighted avg      0.465     0.482     0.465      1586


  PerDay×0.3 + SumLGB×0.3 + Hybrid×0.4
  acc=48.0%  4-AUC=0.7338  3-AUC=0.7407  2-AUC=0.7852


              precision    recall  f1-score   support

     healthy      0.579     0.684     0.627       560
 prediabetes      0.365     0.254     0.299       410
    oral_med      0.441     0.531     0.482       446
     insulin      0.363     0.218     0.272       170

    accuracy                          0.480      1586
   macro avg      0.437     0.422     0.420      1586
weighted avg      0.462     0.480     0.463      1586


  PerDay×0.4 + SumLGB×0.3 + Hybrid×0.3
  acc=48.6%  4-AUC=0.7311  3-AUC=0.7387  2-AUC=0.7836
              precision    recall  f1-score   support

     healthy      0.578     0.693     0.630       560
 prediabetes      0.371     0.256     0.303       410
    oral_med      0.447     0.572     0.502       446
     insulin      0.371     0.135     0.198       170

    accuracy                          0.486      1586
   macro avg      0.442     0.414     0.408      1586
weighted avg      0.466     0.486     0.463      1586


  PerDay×0.5 + SumLGB×0.2 + Hybrid×

              precision    recall  f1-score   support

     healthy      0.560     0.679     0.614       560
 prediabetes      0.357     0.241     0.288       410
    oral_med      0.444     0.565     0.498       446
     insulin      0.359     0.135     0.197       170

    accuracy                          0.475      1586
   macro avg      0.430     0.405     0.399      1586
weighted avg      0.454     0.475     0.452      1586



## Final Comparison

In [7]:
print(f"{'Model':<55} {'Acc':>7} {'4-AUC':>7} {'3-AUC':>7} {'2-AUC':>7}")
print('=' * 85)
print(f"{'BASELINE: LSTM distilled (wearable only)':<55} {'37.6%':>7} {'0.6879':>7} {'0.6858':>7} {'0.6846':>7}")
print(f"{'Hybrid LSTM (wearable+survey, distilled)':<55} {'43.3%':>7} {'0.7167':>7} {'0.7213':>7} {'0.7474':>7}")
print(f"{'LGB summary-stats (prev notebook)':<55} {'46.3%':>7} {'0.7024':>7} {'0.7171':>7} {'0.7658':>7}")
print(f"{'Prev best ensemble (LGB×0.4 + Hybrid×0.6)':<55} {'47.0%':>7} {'0.7333':>7} {'0.7394':>7} {'0.7823':>7}")
print('-' * 85)
print(f"{'LGB per-day+agg+survey (4-class)':<55} {acc4_a*100:>6.1f}% {auc4_a:>7.4f} {auc3_a:>7.4f} {auc2_a:>7.4f}")
print(f"{'LGB per-day+agg+survey (3-class dedicated)':<55} {'—':>7} {'—':>7} {auc3_ded_a:>7.4f} {'—':>7}")
print(f"{'LGB per-day+agg+survey (2-class dedicated)':<55} {'—':>7} {'—':>7} {'—':>7} {auc2_ded_a:>7.4f}")
print(f"{'LGB per-day only (4-class)':<55} {acc4_b*100:>6.1f}% {auc4_b:>7.4f} {auc3_b:>7.4f} {auc2_b:>7.4f}")
print(f"{'LGB per-day only (3-class dedicated)':<55} {'—':>7} {'—':>7} {auc3_ded_b:>7.4f} {'—':>7}")
print(f"{'LGB per-day only (2-class dedicated)':<55} {'—':>7} {'—':>7} {'—':>7} {auc2_ded_b:>7.4f}")

Model                                                       Acc   4-AUC   3-AUC   2-AUC
BASELINE: LSTM distilled (wearable only)                  37.6%  0.6879  0.6858  0.6846
Hybrid LSTM (wearable+survey, distilled)                  43.3%  0.7167  0.7213  0.7474
LGB summary-stats (prev notebook)                         46.3%  0.7024  0.7171  0.7658
Prev best ensemble (LGB×0.4 + Hybrid×0.6)                 47.0%  0.7333  0.7394  0.7823
-------------------------------------------------------------------------------------
LGB per-day+agg+survey (4-class)                          47.9%  0.7067  0.7213  0.7690
LGB per-day+agg+survey (3-class dedicated)                    —       —  0.7266       —
LGB per-day+agg+survey (2-class dedicated)                    —       —       —  0.7685
LGB per-day only (4-class)                                47.8%  0.7066  0.7199  0.7683
LGB per-day only (3-class dedicated)                          —       —  0.7333       —
LGB per-day only (2-class dedicate